In [128]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"C:\Users\ahmed\Desktop\housing_dirty_600.csv")

In [152]:
print(df.isnull().sum())
df.head()                              
#print(df.describe())

price                       0
area                        0
bedrooms                    0
bathrooms                   0
stories                     0
mainroad                    0
guestroom                   0
basement                    0
hotwaterheating             0
airconditioning             0
parking                     0
prefarea                    0
furnishingstatus            0
price_missing               0
area_missing                0
bedrooms_missing            0
furnishingstatus_missing    0
dtype: int64


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus,price_missing,area_missing,bedrooms_missing,furnishingstatus_missing
2,11831351.0,8842.0,1.0,1,2,0,1,0,1,0,0,0,2,0,0,0,0
3,6054572.0,12780.0,1.0,3,1,0,1,0,0,1,2,0,2,0,0,0,0
4,3984489.0,7238.0,1.0,3,2,0,0,0,0,0,0,1,2,0,0,0,0
5,11708614.0,9301.0,1.0,3,3,0,1,0,1,0,1,0,2,0,0,0,0
6,11274682.0,5286.0,1.0,1,2,0,0,1,1,1,0,1,2,0,0,0,0


In [132]:
# Remove $ and other non-numeric characters
df['price'] = df['price'].astype(str).str.replace(r'[^0-9.-]', '', regex=True)
# Convert to float, invalid parsing becomes NaN
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df[df['price'] >= 0]
# Now safely compute median and fill NaNs
df['price'] = df['price'].fillna(df['price'].median())

print(df['price'].isnull().sum())  # should be 0 after imputation
print(df['price'].describe())  # summary stats

0
count    5.810000e+02
mean     7.431019e+06
std      3.434593e+06
min      1.789353e+06
25%      4.484296e+06
50%      7.265989e+06
75%      1.036147e+07
max      1.347630e+07
Name: price, dtype: float64


In [134]:
# Remove 'sqft' and other non-numeric characters
df['area'] = df['area'].astype(str).str.replace(r'[^0-9.-]', '', regex=True)
# Convert to float, invalid parsing becomes NaN
df['area'] = pd.to_numeric(df['area'], errors='coerce')
df = df.dropna(subset=['area'])  

In [136]:
# Impute with median (robust against outliers)
df['price'] = df['price'].fillna(df['price'].median())
df['area'] = df['area'].fillna(df['area'].median())
df['bedrooms'] = df['bedrooms'].fillna(df['bedrooms'].median())
#drop null in furnishingstatus
df = df.dropna(subset=['furnishingstatus'])  
# Create indicator columns
df['price_missing'] = df['price'].isnull().astype(int)
df['area_missing'] = df['area'].isnull().astype(int)
df['bedrooms_missing'] = df['bedrooms'].isnull().astype(int)
df['furnishingstatus_missing'] = df['furnishingstatus'].isnull().astype(int)
df = df[df['bedrooms'] <= 13]

In [138]:
# Check duplicates
print(df.duplicated().sum())
df = df.drop_duplicates()

7


In [140]:
# Check types
print(df.dtypes)
# Check
print(df['bedrooms'].unique())
print(df['bedrooms'].isnull().sum())

df = df.dropna(subset=['area'])  

price                       float64
area                        float64
bedrooms                    float64
bathrooms                     int64
stories                       int64
mainroad                     object
guestroom                    object
basement                     object
hotwaterheating              object
airconditioning              object
parking                       int64
prefarea                     object
furnishingstatus             object
price_missing                 int32
area_missing                  int32
bedrooms_missing              int32
furnishingstatus_missing      int32
dtype: object
[1. 2. 5. 3. 4.]
0


In [142]:
print(df['furnishingstatus'].unique())
df['furnishingstatus'] = df['furnishingstatus'].astype(str).str.strip().str.lower()
df['furnishingstatus'] = df['furnishingstatus'].replace({
    'furnisheeed': 'furnished',
    'semi-furnished': 'semi-furnished',
    'unfurnished': 'unfurnished',
    'furnished': 'furnished'
})
mapping = {'unfurnished':0, 'semi-furnished':1, 'furnished':2}
df['furnishingstatus'] = df['furnishingstatus'].map(mapping)

['furnisheeed' 'furnished' 'semi-furnished' 'unfurnished']


In [144]:
print(df[['bathrooms','stories']].isnull().sum())
for col in ['mainroad','guestroom','basement','hotwaterheating','airconditioning','prefarea']:
    print(df[col].unique())
binary_map = {
    'yes' : 1,
    'y' : 1,
    'yep' : 1,
    'YES' : 1,
    'Yes' : 1,
    'no' : 0,
    'n' : 0,
    'nop' : 0,
    'NO' : 0,
    'No' : 0
}
for col in ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']:
    df[col] = df[col].astype(str).str.strip().str.lower()
    df[col] = df[col].map(binary_map)

bathrooms    0
stories      0
dtype: int64
['nop' 'no' 'yes' 'N' 'Yes' 'No' 'YES' 'Y' 'yep' 'NO']
['yes' 'no' 'Yes' 'YES' 'N' 'Y' 'NO' 'No' 'yep' 'nop']
['no' 'NO' 'yes' 'Y' 'Yes' 'nop' 'yep' 'YES' 'N' 'No']
['yes' 'no']
['no' 'yes']
['no' 'yes']


In [156]:
df.to_csv("cleaned_dataset.csv", index=False)
import os
print(os.getcwd())

C:\Users\ahmed
